<h1>MonReader: Single Frame Modelling (SFM): Model Training

In [10]:
from pathlib import Path
PROJ_ROOT = Path().resolve().parents[0]
DATA_DIR = PROJ_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
TRAIN_DATA_DIR = RAW_DATA_DIR / "training"
TEST_DATA_DIR = RAW_DATA_DIR / "testing"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
SFM_DATA_DIR = PROCESSED_DATA_DIR / "sfm"

MODELS_DIR = PROJ_ROOT / "models"
SFM_MODELS_DIR = MODELS_DIR / "sfm"

In [3]:
import pickle

pkl_data = {}

for pkl_file in SFM_DATA_DIR.glob("*.pkl"):
    with pkl_file.open("rb") as file:
        pkl_data[pkl_file.stem] = pickle.load(file)

print(f"Loaded {len(pkl_data)} pickle files:")
print(list(pkl_data))

Loaded 4 pickle files:
['X_test', 'X_train', 'y_test', 'y_train']


In [7]:
X_train = pkl_data['X_train']
y_train = pkl_data['y_train']
X_test = pkl_data['X_test']
y_test = pkl_data['y_test']

Using Tensorflow models

In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D

Next is to build a starter CNN model. Below is a model that I made for a previous computer vision classification project which served well as a starter model:

In [9]:
model = Sequential([
# First Convolutional Block
Conv2D(64, (3, 3), activation='relu', input_shape=X_train[1].shape), #input shape = (h, w, channel) i.e. image size + channel
BatchNormalization(), #Normalizes the layer activations per batch (helps to speed training)
Conv2D(64, (3, 3), activation='relu'),
MaxPooling2D(pool_size=(2, 2)), #Reduces spatial dimension by 2 (H x W -> H/2 x w/2). Reduces computation
Dropout(0.3),

# Second Convolutional Block
Conv2D(64, (3, 3), activation='relu'),
BatchNormalization(),
Conv2D(128, (3, 3), activation='relu'),
MaxPooling2D(pool_size=(2, 2)),
Dropout(0.3),

# Flatten and Dense layers
Flatten(),
Dense(64, activation='relu'),
BatchNormalization(),
Dropout(0.5),
Dense(2, activation='softmax')  # 2 classes for the target
])

# Compile the model
model.compile(
optimizer='adam',
loss='categorical_crossentropy',
metrics=[tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), "accuracy"]
)
# Display model summary
model.summary()

c:\Users\cochr\Data Science Projects\gejIeSjwpDqobLkc\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 478, 268, 64)   │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 478, 268, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 476, 266, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 238, 133, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 238, 133, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 236, 131, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 236, 131, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 234, 129, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 117, 64, 128)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 117, 64, 128)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 958464)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │    61,341,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,491,010 (234.57 MB)

 Trainable params: 61,490,626 (234.57 MB)

 Non-trainable params: 384 (1.50 KB)